# CatBoost Tuning

This notebook focuses on systematic hyperparameter tuning of CatBoost.

Previous experiments were used to understand the behavior of individual
hyperparameters. Here, the goal is to search for strong combinations of
hyperparameters using a reproducible cross-validation procedure.

## Goals

- Define a reasonable CatBoost search space.
- Tune interacting hyperparameters jointly.
- Use early stopping to control the number of boosting iterations.
- Track cross-validation results.
- Select promising configurations.
- Evaluate the final configuration without data leakage.

In [ ]:
#%cd ..

%load_ext autoreload
%autoreload 2

c:\Users\coffe\Desktop\titanic-ml\titanic-ml


In [6]:
import optuna

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

from src.catboost_validation import evaluate_catboost

SEED = 42

c:\Users\coffe\miniconda3\envs\titanic-ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
train = pd.read_csv("data/train.csv")

X = train.drop(columns="Survived")
y = train["Survived"]

CAT_FEATURES = ["Sex", "Embarked", "Pclass"]

In [11]:
def objective(trial):
    params = {
        'iterations': 3000,
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 50.0, log=True),
        'random_strength': trial.suggest_float("random_strength", 0.0, 5.0),
    }
    
    result = evaluate_catboost(
        X=X,
        y=y,
        cat_features=CAT_FEATURES,
        model_params=params,
        early_stopping_rounds=100,
        seed=SEED,
    )
    
    trial.set_user_attr(
        'cv_std',
        result['std'],
    )
    
    trial.set_user_attr(
        'mean_best_iteration',
        result['mean_best_iteration'],
    )
    
    return result['mean']

In [13]:
sampler = optuna.samplers.TPESampler(seed=SEED)

study = optuna.create_study(direction='maximize', sampler=sampler)

study.optimize(objective, n_trials=30)

print(study.best_value)
print(study.best_params)

[I 2026-09-29 19:42:17,926] A new study created in memory with name: no-name-d28d37f2-2d30-42ec-b709-7e33fce28cfa
[I 2026-09-29 19:43:18,613] Trial 0 finished with value: 0.8394702153034963 and parameters: {'depth': 5, 'learning_rate': 0.17254716573280354, 'l2_leaf_reg': 17.524101118128137, 'random_strength': 2.993292420985183}. Best is trial 0 with value: 0.8394702153034963.
[I 2026-09-29 19:45:39,692] Trial 1 finished with value: 0.8293766869625259 and parameters: {'depth': 4, 'learning_rate': 0.015957084694148364, 'l2_leaf_reg': 1.2551115172973832, 'random_strength': 4.330880728874676}. Best is trial 0 with value: 0.8394702153034963.
[I 2026-09-29 19:46:41,624] Trial 2 finished with value: 0.833864791915134 and parameters: {'depth': 7, 'learning_rate': 0.08341106432362087, 'l2_leaf_reg': 1.0838581269344745, 'random_strength': 4.8495492608099715}. Best is trial 0 with value: 0.8394702153034963.
[I 2026-09-29 19:48:29,606] Trial 3 finished with value: 0.8260184545853996 and parameters

0.843977151465696
{'depth': 6, 'learning_rate': 0.10508421338691762, 'l2_leaf_reg': 2.1839352923182975, 'random_strength': 2.571172192068058}


In [14]:
trials_df = study.trials_dataframe()

trials_df.sort_values(
    "value",
    ascending=False,
).head(10)

,number,value,datetime_start,datetime_complete,duration,params_depth,params_l2_leaf_reg,params_learning_rate,params_random_strength,user_attrs_cv_std,user_attrs_mean_best_iteration,system_attrs_tpe:relative_params:0,state
6,6,0.843977,2026-09-29 19:52:09.196773,2026-09-29 19:52:50.929246,0 days 00:00:41.732473,6,2.183935,0.105084,2.571172,0.016965,85.6,NaN,COMPLETE
25,25,0.843971,2026-09-29 20:14:54.078724,2026-09-29 20:15:42.238834,0 days 00:00:48.160110,8,6.237451,0.136140,2.970583,0.016250,74.4,"{""depth"": 8, ""l2_leaf_reg"": 6.237451098027182,...",COMPLETE
13,13,0.843965,2026-09-29 20:05:05.692291,2026-09-29 20:05:42.537784,0 days 00:00:36.845493,7,2.598895,0.194175,2.181200,0.018809,40.4,"{""depth"": 7, ""l2_leaf_reg"": 2.5988950935141806...",COMPLETE
16,16,0.842835,2026-09-29 20:06:59.970460,2026-09-29 20:07:52.278281,0 days 00:00:52.307821,7,6.244433,0.103614,2.256282,0.022956,108.4,"{""depth"": 7, ""l2_leaf_reg"": 6.244432653846886,...",COMPLETE
12,12,0.841724,2026-09-29 20:03:28.355957,2026-09-29 20:05:05.691288,0 days 00:01:37.335331,5,13.291364,0.040170,1.621203,0.015062,443.2,"{""depth"": 5, ""l2_leaf_reg"": 13.29136371042208,...",COMPLETE
15,15,0.840613,2026-09-29 20:06:26.200872,2026-09-29 20:06:59.969460,0 days 00:00:33.768588,7,3.060606,0.192607,2.505110,0.017024,32.6,"{""depth"": 7, ""l2_leaf_reg"": 3.0606060236915864...",COMPLETE
4,4,0.839489,2026-09-29 19:48:29.608342,2026-09-29 19:49:38.271865,0 days 00:01:08.663523,5,5.418282,0.048164,1.456146,0.017035,259.4,NaN,COMPLETE
9,9,0.839483,2026-09-29 19:57:29.088796,2026-09-29 20:01:16.303788,0 days 00:03:47.214992,5,14.537556,0.013399,2.200762,0.019818,1218.2,NaN,COMPLETE
0,0,0.839470,2026-09-29 19:42:17.928579,2026-09-29 19:43:18.613106,0 days 00:01:00.684527,5,17.524101,0.172547,2.993292,0.025455,184.2,NaN,COMPLETE
24,24,0.837242,2026-09-29 20:14:17.020386,2026-09-29 20:14:54.076715,0 days 00:00:37.056329,6,3.659705,0.136695,0.948898,0.017923,72.2,"{""depth"": 6, ""l2_leaf_reg"": 3.65970452995977, ...",COMPLETE


In [ ]:
best_params = {
    "depth": 6,
    "learning_rate": 0.105,
    "l2_leaf_reg": 2.18,
    "random_strength": 2.57,
}